# 05. FLUX.1-dev 이미지 생성 — RTX 4090 최적화

RunPod의 NVIDIA RTX 4090(24GB VRAM)에서 [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)를 **로컬 파일**로 실행한다. 학습이 아니라 추론 실습이다.

**속도 최적화:** 기본 설정은 Transformer와 T5 텍스트 인코더를 bitsandbytes **8비트 양자화**하여 GPU에 상주시킨다. 기존의 `sequential CPU offload`보다 GPU↔CPU 가중치 왕복을 줄이는 것을 목표로 한다. 모델 가중치 원본 파일을 다시 다운로드하거나 덮어쓸 필요는 없다.
8비트 양자화는 수치 표현을 바꾸므로 BF16과 결과가 완전히 일치하지 않을 수 있다. **4090에서의 실제 생성 속도와 VRAM은 아직 검증하지 않았다.**

- 모델 디렉터리: `/workspace/models/FLUX.1-dev/` (Hugging Face 접근 권한이 있는 PC 또는 일회용 토큰으로 사전 다운로드)
- RunPod에서는 로그인할 필요 없음. `local_files_only=True` 사용
- 기본 해상도 1024×1024, 30 steps
- 기본 `load_mode="8bit_gpu"`; 실패 시 `"bf16_sequential"`로 되돌릴 수 있음

## 패키지 준비

아래 명령을 **현재 Jupyter 커널의 임시 셀**에서 한 번 실행하고 커널을 재시작한다.

    %pip install "diffusers==0.35.1" "transformers==4.56.2" "accelerate==1.10.1" "huggingface_hub==0.36.2" "bitsandbytes==0.48.1" "safetensors>=0.4.3" "sentencepiece==0.2.1" "protobuf>=3.20.2,<6" Pillow

기존 PyTorch와 CUDA는 RunPod의 설치본을 사용한다. 인증 토큰은 노트북에 넣지 않는다.

In [ ]:
import gc
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version

import torch
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBitsAndBytesConfig
from IPython.display import display

assert torch.cuda.is_available(), "CUDA 지원 PyTorch가 있는 GPU 커널을 선택하세요."
device = torch.device("cuda:0")
torch.cuda.set_device(device)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(device))
free, total = torch.cuda.mem_get_info(device)
print(f"VRAM free/total: {free / 2**30:.2f} / {total / 2**30:.2f} GiB")
print({name: version(name) for name in ["diffusers", "transformers", "accelerate", "bitsandbytes"]})

cache_dir = Path(".cache/huggingface")
output_dir = Path("runs/05_image_generation")
cache_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

## 1. 모델을 8비트로 로딩

`load_mode="8bit_gpu"`에서는 가중치의 **원본 BF16 safetensors를 읽으면서 메모리에 8비트로 로딩**한다.
양자화 대상은 계산량이 큰 **FLUX Transformer와 T5 텍스트 인코더**다. CLIP과 VAE는 원래 정밀도를 유지한다.
준비된 디렉터리 안에 `model_index.json`, `transformer/`, `text_encoder_2/`, `vae/` 등이 있어야 한다.

성공하면 `pipe.to("cuda")`로 GPU에서 실행한다. GPU 메모리가 부족하거나 bitsandbytes 호환 오류가 발생하면 커널을 재시작하고 `load_mode="bf16_sequential"`로 전환한다. 이 모드는 매우 느리지만 기존과 동일한 메모리 절약 방식이다.

**모델 가중치 약 34GB는 디스크에 그대로 있다.** 추가 다운로드나 서버 측 Hugging Face 로그인은 하지 않는다.

In [ ]:
model_id = "/workspace/models/FLUX.1-dev"
revision = None  # 로컬 가중치이므로 온라인 모델 리비전 조회 생략
model_path = Path(model_id)
assert (model_path / "model_index.json").is_file(), f"모델 파일을 찾을 수 없습니다: {model_id}"

load_mode = "8bit_gpu"  # 기본: 빠른 8비트 GPU 상주 / 대안: "bf16_sequential"
assert load_mode in {"8bit_gpu", "bf16_sequential"}

if "pipe" in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()

def load_flux():
    if load_mode == "8bit_gpu":
        t5 = T5EncoderModel.from_pretrained(
            str(model_path / "text_encoder_2"),
            quantization_config=TransformersBitsAndBytesConfig(load_in_8bit=True),
            torch_dtype=torch.bfloat16,
            local_files_only=True,
        )
        transformer = FluxTransformer2DModel.from_pretrained(
            str(model_path / "transformer"),
            quantization_config=DiffusersBitsAndBytesConfig(load_in_8bit=True),
            torch_dtype=torch.bfloat16,
            local_files_only=True,
        )
        pipeline = FluxPipeline.from_pretrained(
            str(model_path),
            transformer=transformer,
            text_encoder_2=t5,
            torch_dtype=torch.bfloat16,
            use_safetensors=True,
            local_files_only=True,
        )
        pipeline.to(device)
    else:
        pipeline = FluxPipeline.from_pretrained(
            str(model_path),
            torch_dtype=torch.bfloat16,
            use_safetensors=True,
            local_files_only=True,
        )
        pipeline.enable_sequential_cpu_offload()
    return pipeline

pipe = load_flux()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
offload_mode = "none" if load_mode == "8bit_gpu" else "sequential"

print("Load mode:", load_mode)
print("Scheduler:", type(pipe.scheduler).__name__)
print("Transformer dtype:", pipe.transformer.dtype)
print("Allocated MiB:", torch.cuda.memory_allocated(device) / 2**20)

## 2. 생성 설정 바꾸기

- `prompt`: 만들고 싶은 장면. 기본값은 도서관에서 책 읽는 로봇.
- `seed`: 초기 노이즈의 난수 시드. 같은 모델과 설정을 비교할 때 고정한다.
- `steps`: 노이즈 제거 반복 횟수. 모델 학습 epoch가 아니다.
- `guidance_scale`: 프롬프트의 영향을 조절한다. FLUX.1-dev 기본 실습값은 3.5.
- `width`, `height`: 출력 해상도. 기본 1024×1024. 해상도가 높을수록 메모리 및 시간이 늘어난다.
- `max_sequence_length`: 텍스트 인코더가 처리할 프롬프트 최대 길이.

FLUX.1-dev는 CLIP·T5 텍스트 인코더, 확산 Transformer, FlowMatch 계열 스케줄러, VAE를 사용한다.
이 노트북에서는 `negative_prompt`와 SD 1.5 전용 `DDIMScheduler`를 사용하지 않는다.
중간 latent 텐서의 실제 shape는 아래 callback에서 출력한다.

In [ ]:
prompt = "a small robot reading a book in a cozy library, warm light, detailed digital art"
seed = 42
steps = 30
guidance_scale = 3.5
width = 1024
height = 1024
max_sequence_length = 512

assert width > 0 and height > 0 and width % 16 == 0 and height % 16 == 0
assert steps > 0

## 3. 이미지 생성, 시간·VRAM 측정 및 선택적 latent 관찰

매 호출마다 generator를 새로 생성하여 seed를 고정한다.
`observe=True`이면 1회차, 매 5회차, 마지막 step에서 latent shape·표준편차를 출력한다.
속도 비교를 위해 첫 생성의 기본값은 `observe=False`로 둔다.

시간은 모델 로딩·다운로드·PNG 저장을 제외한 생성 호출을 측정한다.
VRAM 수치는 PyTorch CUDA allocator 기준이며 전체 드라이버 메모리와는 다를 수 있다.

In [ ]:
def generate(prompt, seed, steps, guidance_scale, width, height, observe=False):
    assert steps > 0 and width > 0 and height > 0 and width % 16 == 0 and height % 16 == 0
    observations = []

    def on_step_end(pipeline, step_index, timestep, callback_kwargs):
        if step_index == 0 or (step_index + 1) % 5 == 0 or step_index + 1 == steps:
            latents = callback_kwargs["latents"]
            row = {
                "step": step_index + 1,
                "timestep": float(timestep.item()) if isinstance(timestep, torch.Tensor) else float(timestep),
                "shape": list(latents.shape),
                "std": latents.float().std().item(),
            }
            observations.append(row)
            print(row)
        return callback_kwargs  # 관찰만 하고 모델의 latent는 변경하지 않음

    kwargs = {}
    if observe:
        kwargs = {
            "callback_on_step_end": on_step_end,
            "callback_on_step_end_tensor_inputs": ["latents"],
        }

    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    with torch.inference_mode():
        result = pipe(
            prompt=prompt,
            generator=torch.Generator(device="cpu").manual_seed(seed),
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            width=width,
            height=height,
            max_sequence_length=max_sequence_length,
            **kwargs,
        )
    torch.cuda.synchronize(device)
    seconds = time.perf_counter() - start

    metrics = {
        "prompt": prompt, "seed": seed, "steps": steps,
        "guidance_scale": guidance_scale, "width": width, "height": height,
        "max_sequence_length": max_sequence_length, "seconds": seconds,
        "allocated_mib": torch.cuda.memory_allocated(device) / 2**20,
        "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20,
        "reserved_mib": torch.cuda.memory_reserved(device) / 2**20,
        "model_id": model_id, "model_revision": revision,
        "scheduler": type(pipe.scheduler).__name__, "offload_mode": offload_mode,
        "load_mode": load_mode,
        "gpu": torch.cuda.get_device_name(device), "torch": torch.__version__,
        "cuda": torch.version.cuda, "diffusers": version("diffusers"),
        "transformers": version("transformers"), "accelerate": version("accelerate"),
        "dtype": str(pipe.transformer.dtype), "observations": observations,
    }
    return result.images[0], metrics

## 4. 첫 이미지 생성 및 저장

기본 프롬프트로 1024×1024 이미지 한 장을 생성한다.
속도 측정값이 불필요한 callback의 영향을 받지 않도록 `observe=False`로 실행한다.
생성 이미지와 설정 JSON은 `runs/05_image_generation/`에 저장한다.

In [ ]:
image, metrics = generate(prompt, seed, steps, guidance_scale, width, height, observe=False)
display(image)
print(f"생성 시간: {metrics['seconds']:.2f}s, 최대 할당 VRAM: {metrics['peak_allocated_mib']:.1f} MiB")

run_dir = output_dir / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
image.save(run_dir / "image.png")
(run_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 위치:", run_dir.resolve())

## 5. 선택 실험: 동일 seed에서 20 / 30 / 40 steps 비교

`run_comparison=False`가 기본값이다. `True`로 변경하면 워밍업 후 세 장을 순서대로 생성한다.
동일한 프롬프트·시드·해상도를 유지하고 steps만 바꿔 속도와 품질을 비교한다.

In [ ]:
run_comparison = False
if run_comparison:
    warmup_image, _ = generate(prompt, seed, 2, guidance_scale, width, height)
    del warmup_image
    comparison_dir = output_dir / ("compare_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    comparison_dir.mkdir(parents=True, exist_ok=False)
    rows = []
    for step_count in [20, 30, 40]:
        sample, row = generate(prompt, seed, step_count, guidance_scale, width, height)
        display(sample)
        print(f"steps={step_count}, seconds={row['seconds']:.2f}, peak MiB={row['peak_allocated_mib']:.1f}")
        sample.save(comparison_dir / f"steps_{step_count}.png")
        rows.append(row)
    (comparison_dir / "metrics.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print("비교 결과:", comparison_dir.resolve())

## 직접 실험 / 문제 해결

1. 동일한 seed로 `warm light`와 `moonlight` 프롬프트를 비교한다.
2. seed를 42와 43으로 바꿔 구도를 비교한다.
3. `guidance_scale` 2.5 / 3.5 / 5.0 결과를 비교한다.
4. `load_mode="8bit_gpu"`와 `"bf16_sequential"`의 측정 시간을 비교한다. 모드 전환 시 커널을 재시작한다.

- **bitsandbytes import 오류**: 첫 안내의 패키지 설치 셀을 실행하고 커널을 재시작한다.
- **CUDA OOM**: 다른 커널을 종료하고 메모리를 비운 뒤 재시도한다. 8비트에서도 OOM이면 `load_mode="bf16_sequential"`로 전환한다. 해상도를 768×768로 낮추는 방법도 있다.
- **토크나이저 오류**: `sentencepiece`, `protobuf` 설치와 커널 재시작을 확인한다.
- **다운로드 오류**: 로컬 모델 디렉터리가 완성돼 있는지 확인한다. 현재 코드는 온라인 다운로드를 하지 않는다.
- **CPU RAM 부족**: BF16 sequential 모드는 CPU에 모델을 적재하므로 RAM이 많이 필요하다.
- **경고**: `torch_dtype` deprecated 메시지는 즉시 실패를 뜻하지 않는다.
- **참고**: 생성한 이미지와 JSON은 `runs/`에 저장하며 Git 커밋 전 Jupyter의 **Clear All Outputs**를 사용한다.

In [ ]:
release_gpu = False  # 실습이 끝났을 때 True로 변경
if release_gpu:
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    print("파이프라인 해제 완료. 다른 커널이 사용하는 메모리는 별도입니다.")

## 참고 / 검증 범위

- [Diffusers — FLUX pipeline](https://huggingface.co/docs/diffusers/v0.35.1/api/pipelines/flux)
- [Diffusers — bitsandbytes quantization](https://huggingface.co/docs/diffusers/v0.35.1/quantization/bitsandbytes)
- [FLUX.1-dev 모델 카드](https://huggingface.co/black-forest-labs/FLUX.1-dev)

기존 BF16 sequential CPU offload 방식에 비해 **CPU↔GPU 전송 병목을 줄이도록** 8비트 GPU 실행 경로를 추가했다.
코드 구조는 검토했지만 **실제 RunPod 4090에서 로딩·추론 속도·VRAM은 아직 실측 검증하지 않았다.**